In [4]:
import numpy as np
import pandas as pd
import sys

import os
import gdown 
import zipfile 
from scipy import stats
# Set the default style for seaborn
import seaborn as sns
sns.set(style="whitegrid")      


import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline

from IPython.display import display, Image
import sklearn 



import time
import warnings 

    
# Librerías ML CPU
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Optuna para bayesian optimization
import optuna

# tensor flow 
import tensorflow as Tf
import tensorflow as tf


from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.utils.validation import check_is_fitted

# Configuración de gráficas
sns.set(style='whitegrid')

# Crear carpetas si no existen
os.makedirs('results', exist_ok=True)
os.makedirs('data', exist_ok=True)
os.makedirs("./results/models", exist_ok=True)

# models 
import joblib
from datetime import datetime




AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/nfl-big-data-bowl-2026-prediction/test_input.csv
/kaggle/input/nfl-big-data-bowl-2026-prediction/test.csv
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/nfl_inference_server.py
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/nfl_gateway.py
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/__init__.py
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/core/templates.py
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/core/base_gateway.py
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/core/relay.py
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/core/kaggle_evaluation.proto
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/core/__init__.py
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2.py
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/core/generated/kaggle_evaluati

In [6]:
data_path = '/kaggle/input/nfl-big-data-bowl-2026-prediction/train'# Ruta base donde se almacenan los archivos CSV


In [7]:
input_dfs = []
for week in range(1, 19):
    fname = os.path.join(data_path, f'input_2023_w{week:02d}.csv')
    if os.path.exists(fname):
        input_dfs.append(pd.read_csv(fname))
        print(f'Archivo {fname} cargado. Observaciones: {input_dfs[-1].shape[0]}')
    else:
        print(f'  No se encontró {fname}')

input_df = pd.concat(input_dfs, ignore_index=True) # The input df is a pandas dataframe with all weeks data 

Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/input_2023_w01.csv cargado. Observaciones: 285714
Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/input_2023_w02.csv cargado. Observaciones: 288586
Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/input_2023_w03.csv cargado. Observaciones: 297757
Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/input_2023_w04.csv cargado. Observaciones: 272475
Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/input_2023_w05.csv cargado. Observaciones: 254779
Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/input_2023_w06.csv cargado. Observaciones: 270676
Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/input_2023_w07.csv cargado. Observaciones: 233597
Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/input_2023_w08.csv cargado. Observaciones: 281011
Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/input_2023_w09.csv cargado. Observaciones:

In [8]:
output_dfs = []
for week in range(1, 19):
    fname = os.path.join(data_path, f'output_2023_w{week:02d}.csv')
    if os.path.exists(fname):
        output_dfs.append(pd.read_csv(fname))
        print(f'Archivo {fname} cargado. Observaciones: {input_dfs[-1].shape[0]}')

    else:
        print(f'  No se encontró {fname}')

if output_dfs:
    output_df = pd.concat(output_dfs, ignore_index=True)
else:
    output_df = pd.DataFrame()



Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/output_2023_w01.csv cargado. Observaciones: 254917
Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/output_2023_w02.csv cargado. Observaciones: 254917
Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/output_2023_w03.csv cargado. Observaciones: 254917
Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/output_2023_w04.csv cargado. Observaciones: 254917
Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/output_2023_w05.csv cargado. Observaciones: 254917
Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/output_2023_w06.csv cargado. Observaciones: 254917
Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/output_2023_w07.csv cargado. Observaciones: 254917
Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/output_2023_w08.csv cargado. Observaciones: 254917
Archivo /kaggle/input/nfl-big-data-bowl-2026-prediction/train/output_2023_w09.csv cargado. Obser

In [9]:
print('\nResumen de dimensiones:')
print('input data:', input_df.shape)
print('output data:', output_df.shape)


Resumen de dimensiones:
input data: (4880579, 23)
output data: (562936, 6)


preprocesamiento

In [10]:
def parse_height(h):
    if isinstance(h, str) and '-' in h:
        ft, inch = h.split('-')
        return int(ft) * 12 + int(inch)
    return np.nan


In [11]:

input_df['player_height'] = input_df['player_height'].apply(parse_height)


calculo de edad 

In [12]:
input_df['player_birth_date'] = pd.to_datetime(input_df['player_birth_date'], errors='coerce')
reference_date = pd.to_datetime('2025-11-19')
input_df['age'] = (reference_date - input_df['player_birth_date']).dt.days / 365.25

In [13]:
numerical_features = ['x','y','s','a','o','dir','player_weight','absolute_yardline_number','ball_land_x', 'ball_land_y' , 'player_height', 'age' ]


In [14]:
categorical_features = ['player_position', 'player_side', 'player_role', 'play_direction']


In [15]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [16]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')) 
])

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


In [18]:
target_cols = ['x', 'y']

In [19]:
 
merge_cols = ['game_id', 'play_id', 'nfl_id', 'frame_id']


data_full = input_df.merge(
    output_df[merge_cols + target_cols], # enrutar con el frame especifico a predecir 
    on=merge_cols, # unir por las columnas clave 
    how='inner',
    suffixes=('', '_out')  # deja las features sin sufijo y marca los targets
)


In [20]:
dataToPredict = ['x_out', 'y_out']

In [21]:
X = data_full[numerical_features + categorical_features]
y = data_full[dataToPredict] 

In [22]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, train_size= 0.6,  test_size=0.4, random_state=42)

In [23]:
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp , test_size=0.5, random_state=42)

In [24]:
print('Tamaños de los subconjuntos:')
print('Train :', X_train.shape)
print('Val   :', X_val.shape)
print('Test  :', X_test.shape)


Tamaños de los subconjuntos:
Train : (336255, 16)
Val   : (112085, 16)
Test  : (112086, 16)


In [25]:
# Ajustamos el preprocesador y generamos matrices listas para Keras
preprocessor_dl = preprocessor  # reutiliza la misma configuración definida arriba
preprocessor_dl.fit(X_train)

X_train_proc = preprocessor_dl.transform(X_train)
X_val_proc = preprocessor_dl.transform(X_val)
X_test_proc = preprocessor_dl.transform(X_test)

# Convertir a matrices densas y tipo float32
if hasattr(X_train_proc, "toarray"):
    X_train_proc = X_train_proc.toarray()
    X_val_proc = X_val_proc.toarray()
    X_test_proc = X_test_proc.toarray()

X_train_proc = X_train_proc.astype('float32')
X_val_proc = X_val_proc.astype('float32')
X_test_proc = X_test_proc.astype('float32')

y_train_proc = y_train[['x_out', 'y_out']].to_numpy(dtype='float32')
y_val_proc = y_val[['x_out', 'y_out']].to_numpy(dtype='float32')
y_test_proc = y_test[['x_out', 'y_out']].to_numpy(dtype='float32')

# Submuestreo opcional para reducir costo (ajusta quick_frac=1.0 para entrenamiento completo)
quick_frac = 0.02
if 0 < quick_frac < 1:
    n = int(len(X_train_proc) * quick_frac)
    idx = np.random.permutation(len(X_train_proc))[:max(1000, n)]
    X_train_dl = X_train_proc[idx]
    y_train_dl = y_train_proc[idx]
else:
    X_train_dl = X_train_proc
    y_train_dl = y_train_proc

X_val_dl = X_val_proc
X_test_dl = X_test_proc
y_val_dl = y_val_proc
y_test_dl = y_test_proc

np.savez('data/prep_data_dl.npz',
         X_train=X_train_proc,
         y_train=y_train_proc,
         X_val=X_val_proc,
         y_val=y_val_proc,
         X_test=X_test_proc,
         y_test=y_test_proc)

print(f'Datos DL -> train {X_train_dl.shape}, val {X_val_dl.shape}, test {X_test_dl.shape}')
print(f'NaNs en X_train_dl: {np.isnan(X_train_dl).sum()} | NaNs en y_train_dl: {np.isnan(y_train_dl).sum()}')


Datos DL -> train (6725, 35), val (112085, 35), test (112086, 35)
NaNs en X_train_dl: 0 | NaNs en y_train_dl: 0


In [26]:
data = np.load('data/prep_data_dl.npz')
X_train_dl = data['X_train'].astype('float32')
y_train_dl = data['y_train'].astype('float32')
X_val_dl   = data['X_val'].astype('float32')
y_val_dl   = data['y_val'].astype('float32')
X_test_dl  = data['X_test'].astype('float32')
y_test_dl  = data['y_test'].astype('float32')

# dimensión del vector de características
feature_dim = X_train_dl.shape[1]

# número de épocas y tamaño de lote para todos los modelos
epochs_dl = 100
batch_size_dl = 256

# callbacks comunes: parada temprana y terminación en NaN
common_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.TerminateOnNaN()
]


In [27]:
# MLP denso
mlp = tf.keras.Sequential(name='mlp_denso')
mlp.add(tf.keras.layers.Input(shape=(feature_dim,)))
mlp.add(tf.keras.layers.Dense(128, activation='relu'))
mlp.add(tf.keras.layers.Dense(64, activation='relu'))
mlp.add(tf.keras.layers.Dense(32, activation='relu'))
mlp.add(tf.keras.layers.Dense(2, activation='linear', dtype='float32'))
mlp.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss='mse', metrics=['mae'])

mlp.fit(X_train_dl, y_train_dl,
        validation_data=(X_val_dl, y_val_dl),
        epochs=epochs_dl,
        batch_size=batch_size_dl,
        callbacks=common_callbacks,
        verbose=1)

# Predicciones y métricas
mlp_val_pred  = mlp.predict(X_val_dl,  batch_size=batch_size_dl)
mlp_test_pred = mlp.predict(X_test_dl, batch_size=batch_size_dl)


I0000 00:00:1764789517.074953      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1764789517.075584      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/100


I0000 00:00:1764789519.681207     113 service.cc:148] XLA service 0x78b58401f560 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1764789519.681834     113 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1764789519.681853     113 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1764789519.934432     113 cuda_dnn.cc:529] Loaded cuDNN version 90300


  72/1314 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2056.3040 - mae: 37.2403

I0000 00:00:1764789520.933851     113 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1314/1314 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 361.2345 - mae: 10.2745 - val_loss: 16.4925 - val_mae: 2.9611
Epoch 2/100
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 16.1914 - mae: 2.9341 - val_loss: 15.6389 - val_mae: 2.8648
Epoch 3/100
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 15.5248 - mae: 2.8609 - val_loss: 15.2403 - val_mae: 2.8265
Epoch 4/100
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 15.2530 - mae: 2.8338 - val_loss: 14.8726 - val_mae: 2.7858
Epoch 5/100
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 14.8815 - mae: 2.7976 - val_loss: 15.1434 - val_mae: 2.8353
Epoch 6/100
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 14.6975 - mae: 2.7686 - val_loss: 14.5055 - val_mae: 2.7414
Epoch 7/100
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 14.4573 - mae: 2.7495 - val_loss: 14.7190 - val_mae: 2.7834
Epoch 8/100
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 14.4018 - mae: 2.7419 - val_loss: 14.2306 - val_mae: 2.7079
Epoch 9/100
1314/1314 ━━━━

definimos lo necesario para el tabnet

In [28]:
pip install pytorch-tabnet2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.9/70.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 91.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 73.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 33.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 80.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━

In [29]:
import torch
from pytorch_tabnet import TabNetRegressor

# ------------------------------------------------------------------
# Datos de ejemplo (en tu caso vienen de prep_data_dl.npz)
# ------------------------------------------------------------------
X = np.random.rand(100, 10).astype(np.float32)
y = np.random.rand(100, 1).astype(np.float32)

# ------------------------------------------------------------------
# Definición del modelo TabNet moderno
# ------------------------------------------------------------------
tabnet = TabNetRegressor(
    n_d=16,
    n_a=16,
    n_steps=5,
    gamma=1.3,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=1e-3),
    verbose=1
)


/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/tab_models/tab_reg.py:22: UserWarning: Device used : cuda
  super(TabNetRegressor, self).__post_init__()


y entrenamos

In [30]:
tabnet.fit(
    X_train=X_train_dl, y_train=y_train_dl,
    eval_set=[(X_val_dl, y_val_dl)],
    eval_metric=["mae"],
    max_epochs=100,
    batch_size=1024,
    virtual_batch_size=128,
    patience=20
)


epoch 0  | loss: 2134.70045| val_0_mae: 31.34238|  0:00:20s
epoch 1  | loss: 696.76629| val_0_mae: 11.566  |  0:00:40s
epoch 2  | loss: 114.3093| val_0_mae: 6.01327 |  0:00:59s
epoch 3  | loss: 51.75554| val_0_mae: 4.39833 |  0:01:19s
epoch 4  | loss: 36.11465| val_0_mae: 3.96769 |  0:01:39s
epoch 5  | loss: 30.23033| val_0_mae: 3.67764 |  0:01:58s
epoch 6  | loss: 27.3878 | val_0_mae: 3.56602 |  0:02:18s
epoch 7  | loss: 25.65989| val_0_mae: 3.48202 |  0:02:37s
epoch 8  | loss: 24.59526| val_0_mae: 3.41761 |  0:02:57s
epoch 9  | loss: 23.61196| val_0_mae: 3.35661 |  0:03:17s
epoch 10 | loss: 22.54085| val_0_mae: 3.28715 |  0:03:36s
epoch 11 | loss: 21.951  | val_0_mae: 3.27742 |  0:03:56s
epoch 12 | loss: 21.46133| val_0_mae: 3.24962 |  0:04:16s
epoch 13 | loss: 21.05861| val_0_mae: 3.19766 |  0:04:35s
epoch 14 | loss: 20.72024| val_0_mae: 3.18628 |  0:04:55s
epoch 15 | loss: 20.36983| val_0_mae: 3.2362  |  0:05:14s
epoch 16 | loss: 20.08706| val_0_mae: 3.17249 |  0:05:34s
epoch 17 | 

/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/callbacks/container.py:113: UserWarning: Best weights from best epoch are automatically used!
  callback.on_train_end(logs)


metricas

In [31]:
# Predicciones y métricas
tabnet_val_pred  = tabnet.predict(X_val_dl)
tabnet_test_pred = tabnet.predict(X_test_dl)


In [32]:

def metricas_basicas(y_true, y_pred):
    # MAE
    mae = float(np.mean(np.abs(y_true - y_pred)))

    # MSE
    mse = float(np.mean((y_true - y_pred)**2))

    # RMSE
    rmse = float(np.sqrt(mse))

    # R²
    ss_res = float(np.sum((y_true - y_pred)**2))
    ss_tot = float(np.sum((y_true - np.mean(y_true, axis=0))**2))
    r2 = float(1 - ss_res / (ss_tot + 1e-9))

    # MAPE
    mape = float(np.mean(
        np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true))
    ) * 100)

    return mae, rmse, r2, mape

In [33]:
# Calcula métricas por modelo y conjunto
dl_runs = []
for modelo, preds_val, preds_test in [
    ('MLP denso',        mlp_val_pred,  mlp_test_pred),
    ('tabnet',        tabnet_val_pred,  tabnet_test_pred)
]:
    for tag, y_true, y_pred in [('val', y_val_dl, preds_val), ('test', y_test_dl, preds_test)]:
        mae, rmse, r2, mape = metricas_basicas(y_true, y_pred)
        dl_runs.append([modelo, tag, mae, rmse, r2, mape])

dl_results_df = pd.DataFrame(dl_runs, columns=['Modelo', 'Conjunto', 'MAE', 'RMSE', 'R2', 'MAPE (%)'])


In [34]:
# Selecciona solo métricas de validación de los DL
dl_val = dl_results_df[dl_results_df['Conjunto'] == 'val'].copy()

# Mapea a la estructura que ya usabas en "results"
dl_results_list = [
[
    [row['Conjunto'], row['MAE'], row['RMSE'], row['R2'], row['MAPE (%)']]
    for _, row in dl_results_df[dl_results_df['Conjunto'] == 'val'].iterrows()
]
]

# Elige conjunto (val o test)
dl_scope = dl_results_df[dl_results_df['Conjunto'] == 'val']  # o 'test'

# Construye la tabla final (con nombre de modelo)
results = dl_scope[['Modelo', 'MAE', 'RMSE', 'R2', 'MAPE (%)']].values.tolist()

In [35]:
metrics_df = pd.DataFrame(results, columns=['Modelo', 'MAE', 'RMSE', 'R²', 'MAPE (%)'])
metrics_df = metrics_df.sort_values(by='R²', ascending=False).reset_index(drop=True)

save_path = 'results/model_performance_comparison.csv'
metrics_df.to_csv(save_path, index=False)
display(metrics_df)

,Modelo,MAE,RMSE,R²,MAPE (%)
0,tabnet,2.663215,3.716441,0.966311,11.715603
1,MLP denso,2.683578,3.733400,0.966003,11.645531


In [40]:
model_dict = {
    'tabnet': tabnet

}

best_model_name = metrics_df.iloc[0]['Modelo']  # asumiendo metrics_df solo tiene DL
best_model = model_dict[best_model_name]
print(best_model_name, best_model)


tabnet TabNetRegressor(n_d=16, n_a=16, n_steps=5, gamma=1.3, cat_idxs=[], cat_dims=[], cat_emb_dim=[], n_independent=2, n_shared=2, epsilon=1e-15, momentum=0.02, lambda_sparse=0.0001, seed=0, clip_value=1, verbose=1, optimizer_fn=<class 'torch.optim.adam.Adam'>, optimizer_params={'lr': 0.001}, scheduler_fn=None, scheduler_params={}, mask_type='sparsemax', input_dim=35, output_dim=2, device_name='auto', n_shared_decoder=1, n_indep_decoder=1, grouped_features=[], compile_backend='')


Prueba con el mejor modelo

In [41]:
X_train_full = pd.concat([X_train, X_val], axis=0)
y_train_full = pd.concat([y_train, y_val], axis=0)

In [42]:

# Reutiliza el preprocesador y densifica
X_train_full_proc = preprocessor_dl.transform(X_train_full)
if hasattr(X_train_full_proc, "toarray"):
    X_train_full_proc = X_train_full_proc.toarray()
X_train_full_proc = X_train_full_proc.astype("float32")

y_train_full_proc = y_train_full[['x_out', 'y_out']].to_numpy(dtype='float32')


# Transforma X_test con el mismo preprocesador que usaste para entrenar
X_test_proc = preprocessor_dl.transform(X_test)
if hasattr(X_test_proc, "toarray"):
    X_test_proc = X_test_proc.toarray()
X_test_proc = X_test_proc.astype("float32")

# Si necesitas y_test en numpy para métricas:
y_test_proc = y_test[['x_out', 'y_out']].to_numpy(dtype='float32')


print(X_train_full_proc.dtype, y_train_full_proc.dtype)  # debe ser float32



float32 float32


In [43]:
best_model.fit(X_train_full_proc, y_train_full_proc, batch_size=256, epochs=8)

/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/tab_models/abstract_models/supervised_model.py:137: UserWarning: No early stopping will be performed, last training weights will be used.
  self._set_callbacks(callbacks)


epoch 0  | loss: 655.64734|  0:00:51s
epoch 1  | loss: 33.6508 |  0:01:43s
epoch 2  | loss: 24.39635|  0:02:35s
epoch 3  | loss: 21.85652|  0:03:27s
epoch 4  | loss: 20.75907|  0:04:18s
epoch 5  | loss: 19.88064|  0:05:10s
epoch 6  | loss: 19.1457 |  0:06:02s
epoch 7  | loss: 18.45893|  0:06:54s
epoch 8  | loss: 17.73903|  0:07:46s
epoch 9  | loss: 17.39496|  0:08:38s
epoch 10 | loss: 17.13508|  0:09:30s
epoch 11 | loss: 16.9233 |  0:10:22s
epoch 12 | loss: 16.70689|  0:11:13s
epoch 13 | loss: 16.43158|  0:12:05s
epoch 14 | loss: 16.27057|  0:12:57s
epoch 15 | loss: 16.09579|  0:13:49s
epoch 16 | loss: 16.09372|  0:14:41s
epoch 17 | loss: 15.83344|  0:15:33s
epoch 18 | loss: 15.71154|  0:16:24s
epoch 19 | loss: 15.6794 |  0:17:16s
epoch 20 | loss: 15.50443|  0:18:08s
epoch 21 | loss: 15.43498|  0:19:00s
epoch 22 | loss: 15.4179 |  0:19:52s
epoch 23 | loss: 15.3084 |  0:20:44s
epoch 24 | loss: 15.24251|  0:21:35s
epoch 25 | loss: 15.26528|  0:22:27s
epoch 26 | loss: 15.1502 |  0:23:19s


In [48]:
y_test_pred_best_model = best_model.predict(X_test_proc)


veamos el rendimiento para este caso

In [49]:
mae_best_model = mean_absolute_error(y_test, y_test_pred_best_model)
mse_best_model = mean_squared_error(y_test, y_test_pred_best_model)

rmse_best_model= np.sqrt(mean_squared_error(y_test, y_test_pred_best_model))
r2_best_model = r2_score(y_test, y_test_pred_best_model)
mape_best_model = np.mean(np.abs((y_test - y_test_pred_best_model) / y_test)) * 100


print('\nPrueba best model:')
print(f'MAE : {mae_best_model:.4f}')
print(f'MSE : {mse_best_model:.4f}')
print(f'RMSE: {rmse_best_model:.4f}')
print(f'R2  : {r2_best_model:.4f}')
print(f'MAPE: {mape_best_model:.4f}%')




Prueba best model:
MAE : 2.5228
MSE : 14.0116
RMSE: 3.7432
R2  : 0.9518
MAPE: 10.8365%


creacion formato de entrega .csv

In [50]:
test_input = pd.read_csv('/kaggle/input/nfl-big-data-bowl-2026-prediction/test_input.csv') ### 


In [51]:
test_merged = test_input

In [52]:
test_merged['player_height'] = test_merged['player_height'].apply(parse_height)
test_merged['player_birth_date'] = pd.to_datetime(test_merged['player_birth_date'], errors='coerce')
reference_date = pd.to_datetime('2025-11-19')
test_merged['age'] = (reference_date - test_merged['player_birth_date']).dt.days / 365.25



In [53]:

available_features = [c for c in numerical_features + categorical_features if c in test_merged.columns]
missing_features = [c for c in numerical_features + categorical_features if c not in test_merged.columns]

if missing_features:
    print(f" Faltan columnas: {missing_features}")
    for col in missing_features:
        test_merged[col] = 0  # crear columnas vacías

test_features = test_merged[numerical_features + categorical_features]


In [54]:

expected_input_cols = list(preprocessor.feature_names_in_)

for col in expected_input_cols:
    if col not in test_merged.columns:
        test_merged[col] = 0

test_features = test_merged[expected_input_cols]

print(f" Alineadas: {len(test_features.columns)} columnas (esperadas: {len(expected_input_cols)})")

 Alineadas: 16 columnas (esperadas: 16)


In [55]:
test_features_proc = preprocessor_dl.transform(test_features)
if hasattr(test_features_proc, "toarray"):
    test_features_proc = test_features_proc.toarray()
test_features_proc = test_features_proc.astype("float32")


In [57]:
submissionPreds = best_model.predict(test_features_proc)


creación del dataframe de submission

In [59]:
submission = pd.DataFrame({
    'game_id': test_merged['game_id'],
    'play_id': test_merged['play_id'],
    'nfl_id': test_merged['nfl_id'],
    'frame_id': test_merged['frame_id'],
    'x': submissionPreds[:, 0],
    'y': submissionPreds[:, 1]
})


In [60]:
test = submission 

In [61]:
test = test[['game_id','play_id','nfl_id','frame_id', 'x', 'y']]


In [62]:

submission['id'] = (
    submission['game_id'].astype(str) + '_' +
    submission['play_id'].astype(str) + '_' +
    submission['nfl_id'].astype(str) + '_' +
    submission['frame_id'].astype(str)
)


In [63]:

submission = submission[['id', 'x', 'y']]


In [64]:
submission = submission.drop_duplicates(subset=['id'], keep='last')


In [65]:
os.makedirs('results', exist_ok=True)
submission.to_csv('results/sample_submission.csv', index=False)
submission.to_csv('sample_submission.csv', index=False)
submission.to_csv('submission.csv', index=False)
submission.to_parquet('submission.parquet', index=False)

print(f" Archivo generado con {len(submission)} filas: results/sample_submission.csv AND local")
print(submission.head(10))

 Archivo generado con 49753 filas: results/sample_submission.csv AND local
                       id          x          y
0   2024120805_74_52518_1  88.038658  33.665195
1   2024120805_74_52518_2  88.028587  33.885834
2   2024120805_74_52518_3  88.002892  33.764545
3   2024120805_74_52518_4  88.026833  33.360706
4   2024120805_74_52518_5  88.065498  32.935516
5   2024120805_74_52518_6  88.079628  32.553989
6   2024120805_74_52518_7  88.064857  32.109005
7   2024120805_74_52518_8  88.008423  31.682808
8   2024120805_74_52518_9  87.994156  31.540674
9  2024120805_74_52518_10  88.050911  31.710289


In [66]:
os.makedirs('results', exist_ok=True)
test.to_csv('results/testDlv.csv', index=False)

print(f" Archivo generado con {len(test)} filas: results/test.csv")
print(test.head(10))

 Archivo generado con 49753 filas: results/test.csv
      game_id  play_id  nfl_id  frame_id          x          y
0  2024120805       74   52518         1  88.038658  33.665195
1  2024120805       74   52518         2  88.028587  33.885834
2  2024120805       74   52518         3  88.002892  33.764545
3  2024120805       74   52518         4  88.026833  33.360706
4  2024120805       74   52518         5  88.065498  32.935516
5  2024120805       74   52518         6  88.079628  32.553989
6  2024120805       74   52518         7  88.064857  32.109005
7  2024120805       74   52518         8  88.008423  31.682808
8  2024120805       74   52518         9  87.994156  31.540674
9  2024120805       74   52518        10  88.050911  31.710289


END